# LAB 2 for CS4243: Perspective-Aware Virtual Advertisement Overlay
## Notebook 1 — Keypoints, descriptors, matching, and planar geometry
> **Introduction.** This notebook is part of **LAB 2 for CS4243**. It develops visual intuition for the two
> branches of Stage 1 (Localise) you implement in `overlaylab/keypoints.py` and `overlaylab/geometry.py`, using a
> deterministic subset of the common clip (every eighth frame at 640 px) and synthetic sequences with known
> answers. The parameter sweeps, the DLT conditioning study and the RANSAC iteration study are *explorations* for your
> own understanding (cells marked **EXPLORATION**): they are not report questions.
> Cells marked **ABLATION / SANITY-CHECK / COMPARISON** are the only places where `cv2.SIFT`,
> `cv2.findHomography` and `cv2.warpPerspective` may appear.

## Learning goals
See how the window scale, the sensitivity constant and the NMS mode change *where* corners land; watch the ratio test
reject repetitive structure; separate in-plane rotation (absorbed by the descriptor) from scale and viewpoint change
(not absorbed); inspect the singular values of the DLT system with and without normalisation; and relate RANSAC's
iteration count to the inlier fraction.

## 1. Imports and reproducible configuration
`notebook_config()` fixes the settings used throughout Notebooks 1–3. The reference board is resized by the same factor
as the frames so that the board appears at a comparable scale in both images.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np
import matplotlib.pyplot as plt
import cv2

from overlaylab import data, metrics, visualize as viz
from overlaylab import supplied as sp
from overlaylab.config import notebook_config, replace
from overlaylab.keypoints import harris_response, detect_corners, describe_keypoints, match_descriptors
from overlaylab.geometry import normalise_points, dlt_homography, ransac_homography, transfer_points, warp_artwork
from overlaylab import synthetic as syn

config = notebook_config()
indices, frames, factor = data.load_notebook_subset()
reference_full = data.load_reference()
reference = sp.resize_width(reference_full, int(round(reference_full.shape[1] * factor)))
config.localisation.warp.reference_shape = reference.shape[:2]
artwork = data.load_artwork("ad")
gt_val = data.subset_gt_corners(indices, factor, "validation")
gt_test = data.subset_gt_corners(indices, factor, "test")
print(f"{len(frames)} subset frames of {frames[0].shape[1]}x{frames[0].shape[0]} (scale {factor:.3f}); "
      f"reference {reference.shape[1]}x{reference.shape[0]}; {len(gt_val)} validation / {len(gt_test)} test annotated frames")


#### (Our Code) Report figure setup


In [ ]:
# Keep report figures in the permitted student_files directory.
# The display DPI keeps previews compact; the export DPI makes saved figures clearer.
FIGURES = ROOT / "student_files" / "stage1"
FIGURES.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({"figure.dpi": 100, "savefig.dpi": 160})


## 2. The clip, the reference, and the artwork
Ground-truth corners exist on every 16th frame of the full clip on which the whole poster is visible (26 frames); here
they are drawn on the subset frames that carry them. Validation frames (even annotation index) are the only frames you
may use to choose thresholds. The clip is real hand-held footage (POT-280 "Coke"), so the corners carry annotation
noise of about a pixel; keep that floor in mind when you read corner errors.

In [ ]:
show = [i for i in indices if i in gt_val][:6]
panels = [viz.label(viz.draw_quad(frames[indices.index(i)], (gt_val | gt_test)[i]), f"frame {i} ({'val' if i in gt_val else 'test'})") for i in show]
plt.figure(figsize=(15, 6)); plt.imshow(viz.montage(panels, cols=3)); plt.axis("off"); plt.title("Subset frames with ground-truth board corners")
fig, axes = plt.subplots(1, 2, figsize=(12, 2.4))
axes[0].imshow(reference); axes[0].set_title("rectified reference board"); axes[0].axis("off")
axes[1].imshow(artwork); axes[1].set_title("replacement artwork ad.png"); axes[1].axis("off")
plt.show()

## 3. Harris corners and the two NMS modes
The response map should peak at corners of the poster texture and stay non-positive on edges and flat
regions. Compare how `"window"` and `"radius"` distribute corners, and how many fall on the board versus the background
(use the ground-truth quadrilateral only to *count*, never to initialise anything).

In [ ]:
frame_id = next(i for i in indices if i in gt_val and i >= 150)
frame = frames[indices.index(frame_id)]; gray = sp.to_gray(frame)
R = harris_response(gray, config.localisation.detector)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].imshow(gray, cmap="gray"); axes[0].set_title(f"frame {frame_id}"); axes[0].axis("off")
axes[1].imshow(np.sign(R) * np.log1p(np.abs(R) / (np.abs(R).max() + 1e-9) * 1e3), cmap="coolwarm"); axes[1].set_title("Harris response (signed log)"); axes[1].axis("off")
plt.show()

board = sp.quad_mask(gray.shape, gt_val[frame_id])
rows = []
for mode, radius in [("window", 4), ("radius", 5), ("radius", 9)]:
    for sigma_w in (1.5, 2.0, 3.0):
        det = replace(config.localisation.detector, nms_mode=mode, nms_radius=radius, sigma_w=sigma_w)
        kps, resp = detect_corners(gray, det)
        on_board = board[kps[:, 1].astype(int), kps[:, 0].astype(int)].sum()
        rows.append((mode, radius, sigma_w, len(kps), int(on_board), len(kps) - int(on_board)))
print(f"{'mode':8s} {'r':>2s} {'sigma_w':>7s} {'total':>6s} {'board':>6s} {'background':>10s}")
for r in rows: print(f"{r[0]:8s} {r[1]:2d} {r[2]:7.1f} {r[3]:6d} {r[4]:6d} {r[5]:10d}")
kw, _ = detect_corners(gray, replace(config.localisation.detector, nms_mode="window", nms_radius=4))
kr, _ = detect_corners(gray, replace(config.localisation.detector, nms_mode="radius", nms_radius=9))
plt.figure(figsize=(15, 4.5)); plt.imshow(viz.montage([viz.label(viz.draw_points(frame, kw, radius=2), "window r=4"), viz.label(viz.draw_points(frame, kr, radius=2), "radius r=9")], cols=2)); plt.axis("off"); plt.show()

### EXPLORATION — parameter sweep: window scale, sensitivity constant $\kappa$, NMS mode
Not a report question. Sweep `sigma_w`, `kappa` and the NMS mode and measure two things on a pair of annotated frames:
how many corners land on the board, and how *repeatable* they are (fraction of corners on frame A whose ground-truth
transfer lands within 3 px of a corner on frame B). Decide which setting you keep and why; the choice is yours and goes
into `config.localisation.detector` for the rest of the lab.

In [ ]:
fa, fb = sorted(gt_val)[0], sorted(gt_val)[1]
A_, B_ = frames[indices.index(fa)], frames[indices.index(fb)]
H_ab = syn.homography_from_corners(gt_val[fa], gt_val[fb])          # board plane: frame a -> frame b (ground truth, used only to score)

def repeatability(det_cfg, tol=3.0):
    ka, _ = detect_corners(sp.to_gray(A_), det_cfg); kb, _ = detect_corners(sp.to_gray(B_), det_cfg)
    on_a = sp.quad_mask(A_.shape[:2], gt_val[fa])[ka[:, 1].astype(int), ka[:, 0].astype(int)]
    if on_a.sum() == 0 or len(kb) == 0: return 0.0, int(on_a.sum())
    proj = transfer_points(H_ab, ka[on_a])
    d = np.linalg.norm(proj[:, None, :] - kb[None, :, :], axis=2).min(axis=1)
    return float((d < tol).mean()), int(on_a.sum())

print(f"{'mode':8s} {'sigma_w':>7s} {'kappa':>6s} {'on board':>8s} {'repeat.':>8s}")
for mode in ("window", "radius"):
    for sigma_w in (1.5, 2.0, 3.0):
        for kappa in (0.04, 0.06, 0.10):
            r, n = repeatability(replace(config.localisation.detector, nms_mode=mode, sigma_w=sigma_w, kappa=kappa))
            print(f"{mode:8s} {sigma_w:7.1f} {kappa:6.2f} {n:8d} {r:8.2f}")
# YOUR CODE HERE (optional) — plot repeatability against sigma_w for both NMS modes; pick the setting you keep


#### (Our Code) Exploration: Harris repeatability sweep and setting selection


In [ ]:
sweep_rows = []
for mode in ("window", "radius"):
    for sigma_w in (1.5, 2.0, 3.0):
        for kappa in (0.04, 0.06, 0.10):
            det = replace(config.localisation.detector,
                          nms_mode=mode, sigma_w=sigma_w, kappa=kappa)
            rate, count = repeatability(det)
            sweep_rows.append((mode, sigma_w, kappa, count, rate))

# Compare smoothing scales at each kappa, using the same NMS radius in both modes.
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, mode in zip(axes, ("window", "radius")):
    for kappa in (0.04, 0.06, 0.10):
        values = [row for row in sweep_rows if row[0] == mode and row[2] == kappa]
        ax.plot([row[1] for row in values], [row[4] for row in values],
                "o-", label=f"kappa={kappa:.2f}")
    ax.set_title(f"{mode} NMS (r={config.localisation.detector.nms_radius})")
    ax.set_xlabel("Gaussian window scale sigma_w")
    ax.set_xticks((1.5, 2.0, 3.0))
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
axes[0].set_ylabel("Repeatability (distance < 3 px)")
fig.suptitle(f"Common clip validation frames {fa} -> {fb}, {A_.shape[1]} px wide")

# Keep the highest-repeatability setting; break exact ties by board-corner count.
best = max(sweep_rows, key=lambda row: (row[4], row[3]))
best_mode, best_sigma, best_kappa, best_count, best_rate = best
config.localisation.detector = replace(
    config.localisation.detector,
    nms_mode=best_mode, sigma_w=best_sigma, kappa=best_kappa,
)
axes[("window", "radius").index(best_mode)].plot(
    best_sigma, best_rate, "k*", markersize=14, label="Selected setting",
)
axes[("window", "radius").index(best_mode)].legend()
fig.tight_layout()
plt.show()
print(f"Selected {best_mode} NMS, sigma_w={best_sigma}, kappa={best_kappa:.2f}: "
      f"repeatability {best_rate:.3f}, {best_count} board corners in frame {fa}.")
print("Selection uses this validation pair only: maximise repeatability, "
      "then board-corner count for exact ties. Applied to config.localisation.detector.")
print("These are 640-pixel exploration settings; validate final native-resolution "
      "settings on common-clip validation frames before test/stress reporting.")

### (Our Code) 3.1 Native-resolution validation and Stage 1 freeze
The preceding two-frame sweep is exploratory. The following targeted checks use all 13 common **validation** corner frames at 1280 × 720, with seed 0 and success at 2% of the longer GT diagonal. Feature availability and matching are investigated before changing geometric settings. Ground-truth transforms below score matches only; they never initialise the localiser. Test and stress frames are not evaluated until this configuration has been frozen. The selected configuration is saved to `outputs/stage1_settings.json`.


In [ ]:
from copy import deepcopy
from dataclasses import asdict
import json
import time
from overlaylab.localisation import localise_board, evaluate_audit_frames

# Select parameters using only validation corner annotations from the manifest.
# These frames and the reference retain their native size; the earlier subset is a preview.
native_gt = data.load_gt_corners(split="validation")
native_frames = {i: image for i, image in sp.iter_video(data.COMMON_CLIP, stop=max(native_gt) + 1)
                 if i in native_gt}
native_reference = data.load_reference()
# Rebuild the original baseline on every run, rather than loading a previously
# selected winner and accidentally treating it as the starting configuration.
baseline_config = notebook_config()
baseline_config.localisation.detector = replace(
    baseline_config.localisation.detector, sigma_w=1.5, kappa=0.04,
    nms_mode="radius", nms_radius=5, max_corners=600)
baseline_config.localisation.warp.reference_shape = native_reference.shape[:2]
baseline_config.localisation.success_threshold = 0.02
# Keep each trial result alongside a separate configuration copy so the
# winning index can recover exactly the settings that produced its measurements.
experiment_log = []
experiment_configs = []

def validation_trial(label, candidate):
    started = time.perf_counter()
    # Use the supplied evaluator so failed localisations remain in the success denominator.
    result = evaluate_audit_frames(native_reference, native_frames, native_gt, candidate.localisation)
    # Store settings and per-frame scalars for inspection; omit corner arrays
    # from this JSON log because the comparison needs errors and diagnostic counts.
    record = dict(label=label, settings=candidate.thresholds()["localisation"],
                  summary=result["summary"],
                  per_frame={str(i): {k: v for k, v in row.items() if k != "corners"}
                             for i, row in result["per_frame"].items()},
                  seconds=time.perf_counter() - started)
    experiment_log.append(record)
    experiment_configs.append(deepcopy(candidate))
    s = record["summary"]
    print(f"{label:30s} success {s['success']:.3f} failed {s['failed']:2d} "
          f"median {s['median']*100:.4f}% p90 {s['p90']*100:.4f}% "
          f"({record['seconds']:.1f}s)", flush=True)
    return len(experiment_log) - 1

def validation_rank(index):
    summary = experiment_log[index]["summary"]
    # Tuples are compared left to right. Negate success to prefer larger rates,
    # then prefer fewer missing models, lower finite p90 error and lower finite median error.
    return (-summary["success"], summary["failed"], summary["p90"], summary["median"])

baseline_index = validation_trial("baseline: 600 corners, r=5", baseline_config)
for i, row in experiment_log[baseline_index]["per_frame"].items():
    print("BASELINE FRAME", i, json.dumps(row), flush=True)
# Inspect the three baseline failures before choosing which parameter groups to test.
for i in (80, 240, 448):
    H, corners, pairs, inliers, diag = localise_board(native_reference, native_frames[i], baseline_config.localisation)
    # This GT-derived transform only labels match accuracy for diagnosis.
    # It is never supplied to localise_board or used to repair its prediction.
    H_score, _, _ = dlt_homography(sp.rect_corners(native_reference.shape[1], native_reference.shape[0]), native_gt[i])
    source = diag["keypoints_ref"][pairs[:, 0]]
    destination = diag["keypoints_frame"][pairs[:, 1]]
    scoring_error = np.linalg.norm(transfer_points(H_score, source) - destination, axis=1)
    board = sp.quad_mask(native_frames[i].shape[:2], native_gt[i])
    kp = diag["keypoints_frame"]
    print("DIAGNOSIS", i, "retained ref/frame", diag["n_keypoints_ref"], diag["n_keypoints_frame"],
          "frame keypoints on board", int(board[kp[:, 1].astype(int), kp[:, 0].astype(int)].sum()),
          "matches/inliers", len(pairs), int(inliers.sum()), "iterations", diag["iterations"],
          "GT-scored matches <4px", int((scoring_error < 4).sum()),
          "source extent", np.ptp(source, axis=0).round(1).tolist() if len(source) else [], flush=True)

# The failed native frames have few matches; compare feature availability first.
for radius in (3, 5, 7):
    for count in (600, 800, 1000):
        if radius == 5 and count == 600:
            continue
        # Hold descriptor, matcher and RANSAC settings fixed while varying the detector.
        candidate = deepcopy(baseline_config)
        candidate.localisation.detector = replace(candidate.localisation.detector,
                                                  nms_radius=radius, max_corners=count)
        validation_trial(f"features: {count} corners, r={radius}", candidate)
# Carry the two best feature configurations forward for an interaction check.
feature_indices = sorted(range(len(experiment_log)), key=validation_rank)[:2]
feature_winner = deepcopy(experiment_configs[feature_indices[0]])
# A larger ratio admits less distinctive matches. The mutual check additionally
# requires a frame descriptor to choose the same reference descriptor in reverse.
for ratio in (0.75, 0.80, 0.85, 0.90):
    for mutual in (False, True):
        candidate = deepcopy(feature_winner)
        candidate.localisation.matcher = replace(candidate.localisation.matcher, ratio=ratio, mutual=mutual)
        validation_trial(f"matches: ratio={ratio}, mutual={mutual}", candidate)
matcher_indices = sorted(range(len(experiment_log)), key=validation_rank)[:2]
# Recheck the runner-up feature configuration with the selected match settings.
for feature_index in feature_indices:
    for matcher_index in matcher_indices:
        candidate = deepcopy(experiment_configs[feature_index])
        candidate.localisation.matcher = deepcopy(experiment_configs[matcher_index].localisation.matcher)
        validation_trial(f"interaction f{feature_index}/m{matcher_index}", candidate)
# Select once from validation results, including the interaction checks.
# Test and stress results below must not influence this decision.
selected_index = min(range(len(experiment_log)), key=validation_rank)
selected_config = deepcopy(experiment_configs[selected_index])
# Clear per-run image dimensions when exporting: final runs use native resolution
# and set the reference shape for the current clip rather than saving the preview shape.
selected_config.resize_width = None
selected_config.localisation.warp.reference_shape = None
print("SELECTED", experiment_log[selected_index]["label"], json.dumps(selected_config.thresholds()), flush=True)
print("VALIDATION EXPERIMENT LOG", json.dumps(experiment_log), flush=True)

# Apply the frozen numerical settings to the smaller preview without retuning them.
preview_cfg = deepcopy(selected_config.localisation)
preview_cfg.warp.reference_shape = reference.shape[:2]
preview_validation = evaluate_audit_frames(reference, {i: frames[indices.index(i)] for i in gt_val}, gt_val, preview_cfg)
print("Frozen 640-pixel validation preview", preview_validation["summary"], flush=True)
# The mask and corner manifests have separate splits. Check that downstream
# validation-mask previews initialise, without scoring against any test corner labels.
validation_mask_ids = sorted(data.load_gt_masks("validation"))
mask_preview = {i: fr for i, fr in sp.iter_video(data.COMMON_CLIP, width=data.NOTEBOOK_WIDTH,
                stop=max(validation_mask_ids) + 1) if i in validation_mask_ids}
for i in validation_mask_ids:
    _, _, _, _, diag = localise_board(reference, mask_preview[i], preview_cfg)
    print("Validation-mask availability (no test corner scoring)", i, diag["status"], diag["n_inliers"], flush=True)
    assert diag["status"], f"Stage 3 validation-mask preview {i} cannot initialise"

config = deepcopy(selected_config)
config.resize_width = data.NOTEBOOK_WIDTH
config.localisation.warp.reference_shape = reference.shape[:2]
# Save a dimension-independent snapshot for later assertions.
frozen_localisation = selected_config.thresholds()["localisation"]
(ROOT / "outputs").mkdir(exist_ok=True)
(ROOT / "outputs" / "stage1_settings.json").write_text(json.dumps(selected_config.thresholds(), indent=2) + "\n")
print("Stage 1 settings saved before held-out analysis.")


## 4. Descriptors and the ratio test
Match the reference to the frame and look at the distribution of first-to-second-neighbour ratios. Repeated texture and
letters produce near-identical neighbours: find one match that the ratio test correctly rejects and one
geometrically correct match that it rejects anyway (material for **Question 1**, Matching).

In [ ]:
g_ref = sp.to_gray(reference)
kp_r, _ = detect_corners(g_ref, config.localisation.detector); kp_f, _ = detect_corners(gray, config.localisation.detector)
d_r, keep_r = describe_keypoints(g_ref, kp_r, config.localisation.descriptor); kp_r = kp_r[keep_r]
d_f, keep_f = describe_keypoints(gray, kp_f, config.localisation.descriptor); kp_f = kp_f[keep_f]
loose = replace(config.localisation.matcher, ratio=1.01)
pairs_all, dist_all, ratio_all = match_descriptors(d_r, d_f, loose)
pairs, dist, ratio = match_descriptors(d_r, d_f, config.localisation.matcher)
print(f"{len(d_r)} reference and {len(d_f)} frame descriptors; {len(pairs_all)} nearest neighbours, {len(pairs)} survive the ratio test at {config.localisation.matcher.ratio}")
plt.figure(figsize=(6, 3)); plt.hist(ratio_all, bins=40, color="steelblue"); plt.axvline(config.localisation.matcher.ratio, color="k", ls="--")
plt.xlabel("d1 / d2"); plt.ylabel("count"); plt.title("ratio-test statistic for all nearest neighbours"); plt.show()
# geometric truth for the shown frame lets us label every nearest neighbour as correct or not
H_gt = syn.homography_from_corners(sp.rect_corners(reference.shape[1], reference.shape[0]), gt_val[frame_id])
proj = transfer_points(H_gt, kp_r[pairs_all[:, 0]]); err = np.linalg.norm(proj - kp_f[pairs_all[:, 1]], axis=1)
correct = err < 4.0; accepted = ratio_all < config.localisation.matcher.ratio
print(f"correct nearest neighbours: {correct.sum()}  accepted: {accepted.sum()}  correct&accepted: {(correct&accepted).sum()}  correct&rejected: {(correct&~accepted).sum()}  wrong&accepted: {(~correct&accepted).sum()}")
plt.figure(figsize=(12, 7)); plt.imshow(viz.draw_matches(reference, frame, kp_r, kp_f, pairs_all, correct & accepted)); plt.axis("off")
plt.title("nearest neighbours: green = geometrically correct AND accepted, red = everything else"); plt.show()


#### (Our Code) Ratio-test rejection examples and report figure


In [ ]:
# Show one rejected incorrect neighbour and one rejected correct neighbour.
# The GT transfer test supplies analysis labels, not a new pipeline acceptance rule.
ratio_examples = []
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
for ax, label, eligible in zip(axes, ("Incorrect match correctly rejected", "Correct match rejected"),
                             ((~correct) & (~accepted), correct & (~accepted))):
    candidates = np.flatnonzero(eligible)
    if not len(candidates):
        ax.text(0.5, 0.5, "No example in this validation frame", ha="center")
        ax.axis("off")
        continue
    # Choose the rejected example nearest the accepted side of the ratio threshold.
    index = candidates[np.argmin(ratio_all[candidates])]
    pair = pairs_all[index:index + 1]
    ax.imshow(viz.draw_matches(reference, frame, kp_r, kp_f, pair,
                               np.array([bool(correct[index])])))
    ax.set_title(f"{label}\nratio={ratio_all[index]:.3f}; GT transfer={err[index]:.2f} px")
    ax.axis("off")
    ratio_examples.append(dict(kind=label, frame=frame_id, ratio=float(ratio_all[index]),
                               geometric_error_px=float(err[index]), pair=pair[0].tolist()))
fig.suptitle(f"Common validation frame {frame_id}, 640 px; frozen Harris descriptor, ratio < {config.localisation.matcher.ratio}", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, .90))
fig.savefig(FIGURES / "ratio_rejections.png", bbox_inches="tight")
plt.show()
# Print the exact indices and measurements used in the figure for report cross-checking.
print("REPORT_RATIO_EXAMPLES", json.dumps(ratio_examples))


## 5. Where does matching break? Rotation versus scale and viewpoint
Plot the matching rate against the subset frames. The dominant-orientation normalisation absorbs in-plane rotation;
scale and out-of-plane tilt are not absorbed. **Question 1** asks the same analysis on the two Stage 1 stress clips
(`data.stress_clips(1)`: the **viewpoint** clip, where the camera orbits the poster, and the **scale** clip, where it
approaches and retreats): pick a frame pair on which the rate collapses, separate the in-plane rotation the dominant
orientation absorbs from the viewpoint / scale change it does not, and compare with the labelled SIFT cell below.
Rehearse the analysis here on the common clip first.

In [ ]:
def matching_rate(fr):
    g = sp.to_gray(fr); k, _ = detect_corners(g, config.localisation.detector)
    d, keep = describe_keypoints(g, k, config.localisation.descriptor)
    p, _, _ = match_descriptors(d_r, d, config.localisation.matcher)
    return len(p) / max(len(d_r), 1), len(p)
# Development plots use validation frames only.
rate_indices = sorted(gt_val)
rates = [matching_rate(frames[indices.index(i)])[0] for i in rate_indices]
import json
(ROOT / "outputs").mkdir(exist_ok=True)
(ROOT / "outputs" / "stage1_settings.json").write_text(json.dumps(config.thresholds(), indent=2))
print("Stage 1 choices saved before stress analysis. Do not retune from the stress rows.")
widths = {i: np.linalg.norm(c[1] - c[0]) for i, c in gt_val.items()}
fig, ax = plt.subplots(figsize=(10, 3)); ax.plot(rate_indices, rates, lw=1.2); ax.set_xlabel("frame"); ax.set_ylabel("matches / reference descriptors")
ax2 = ax.twinx(); ax2.plot(list(widths), list(widths.values()), "o", color="orange", ms=3); ax2.set_ylabel("board width on screen (px)", color="orange"); plt.show()
# in-plane rotation test: rotate one frame by 30 degrees and match again
M = cv2.getRotationMatrix2D((gray.shape[1] / 2, gray.shape[0] / 2), 30, 1.0)
rot = cv2.warpAffine(frame, M, (gray.shape[1], gray.shape[0]))
print("matching rate: original %.3f, rotated 30 deg %.3f, downscaled x0.6 %.3f" % (
    matching_rate(frame)[0], matching_rate(rot)[0], matching_rate(cv2.resize(frame, None, fx=0.6, fy=0.6))[0]))
# Stage 1 stress clips (Question 1): matching rate on the annotated frames, same reference, same settings
fig, axes = plt.subplots(1, 2, figsize=(14, 3))
for ax, (factor_name, cid) in zip(axes, data.stress_clips(1).items()):
    video_s, _ = data.clip_paths(cid); gt_s = data.load_diagnostic_corners(cid, scale=factor)
    rates_s = {i: matching_rate(fr)[0] for i, fr in sp.iter_video(video_s, step=data.GT_STRIDE, width=frames[0].shape[1]) if i in gt_s}
    ax.plot(list(rates_s), list(rates_s.values()), "o-", ms=3); ax.set_xlabel(f"{cid} frame"); ax.set_ylabel("matches / reference descriptors"); ax.set_title(f"{factor_name} clip: matching rate")
plt.show()
# YOUR CODE HERE — pick the frame pair where the rate collapses and relate it to the in-plane rotation, scale and keystone of the GT quad


#### (Our Code) Matching-collapse examples on the scale and viewpoint clips


In [ ]:

# Select examples using the frozen matching settings; ground truth is used only for analysis.
if len(d_r) == 0:
    raise ValueError("Run Section 4 first and check that the reference has valid descriptors.")
matching_cases = []
comparison_width = frames[0].shape[1]

def gt_quad_geometry(quad):
    # TL, TR, BR, BL: top-edge angle, apparent scale, and opposite-edge asymmetry.
    q = np.asarray(quad, dtype=float)
    edges = np.roll(q, -1, axis=0) - q
    top, right, bottom, left = np.linalg.norm(edges, axis=1)
    angle = float(np.degrees(np.arctan2(edges[0, 1], edges[0, 0])))
    area = 0.5 * abs(np.dot(q[:, 0], np.roll(q[:, 1], -1))
                     - np.dot(q[:, 1], np.roll(q[:, 0], -1)))
    ref_area = (reference.shape[1] - 1) * (reference.shape[0] - 1)
    apparent_scale = float(np.sqrt(area / max(ref_area, 1e-9)))
    top_bottom = float(top / max(bottom, 1e-9))
    left_right = float(left / max(right, 1e-9))
    keystone = max(top_bottom, 1 / max(top_bottom, 1e-9),
                   left_right, 1 / max(left_right, 1e-9))
    return angle, apparent_scale, top_bottom, left_right, keystone

for capture_factor, clip_id in data.stress_clips(1).items():
    video_path, _ = data.clip_paths(clip_id)
    gt_stress = data.load_diagnostic_corners(clip_id, scale=factor)
    best_case = worst_case = None
    for frame_index, image in sp.iter_video(video_path, step=data.GT_STRIDE, width=comparison_width):
        if frame_index not in gt_stress:
            continue
        rate, count = matching_rate(image)
        candidate = {"frame": frame_index, "image": image, "custom_rate": rate, "n_matches": count}
        if best_case is None or rate > best_case["custom_rate"]:
            best_case = candidate
        if worst_case is None or rate < worst_case["custom_rate"]:
            worst_case = candidate
    if worst_case is None:
        raise ValueError(f"No annotated frames were loaded for {clip_id}.")

    print(f"\n{clip_id} ({capture_factor}, test, {comparison_width} px): "
          "reference-to-frame matching with frozen settings")
    print(f"{'case':7s} {'frame':>5s} {'matches':>7s} {'rate':>7s} {'angle':>8s} "
          f"{'scale':>7s} {'top/btm':>8s} {'left/right':>10s} {'keystone':>9s}")
    fig, case_axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, label, case in zip(case_axes, ("best", "worst"), (best_case, worst_case)):
        corners_gt = gt_stress[case["frame"]]
        angle, scale_gt, tb, lr, keystone = gt_quad_geometry(corners_gt)
        print(f"{label:7s} {case['frame']:5d} {case['n_matches']:7d} {case['custom_rate']:7.3f} "
              f"{angle:8.1f} {scale_gt:7.3f} {tb:8.3f} {lr:10.3f} {keystone:9.3f}")
        keypoints, _ = detect_corners(sp.to_gray(case["image"]), config.localisation.detector)
        descriptors, kept = describe_keypoints(sp.to_gray(case["image"]), keypoints,
                                               config.localisation.descriptor)
        keypoints = keypoints[kept]
        pairs_case, _, _ = match_descriptors(d_r, descriptors, config.localisation.matcher)
        H_gt_case = syn.homography_from_corners(
            sp.rect_corners(reference.shape[1], reference.shape[0]), corners_gt)
        projected = transfer_points(H_gt_case, kp_r[pairs_case[:, 0]])
        correct_case = np.linalg.norm(projected - keypoints[pairs_case[:, 1]], axis=1) < 4.0
        ax.imshow(viz.draw_matches(reference, viz.draw_quad(case["image"], corners_gt),
                                   kp_r, keypoints, pairs_case, correct_case))
        ax.set_title(f"{label}: reference -> frame {case['frame']}\n"
                     f"rate={case['custom_rate']:.3f}, angle={angle:.1f} deg, "
                     f"scale={scale_gt:.2f}, keystone={keystone:.2f}")
        ax.axis("off")
    fig.suptitle(f"{clip_id}, test, {comparison_width} px: best vs worst reference/frame pair\n"
                 "Accepted matches: green = GT error < 4 px; red = other accepted matches")
    fig.tight_layout(rect=(0, 0, 1, 0.87))
    # Export the existing best/worst matching comparison as a report asset.
    fig.savefig(FIGURES / f"matching_{clip_id}.png", bbox_inches="tight")
    plt.show()
    drop = best_case["custom_rate"] - worst_case["custom_rate"]
    print(f"Matching rate falls from {best_case['custom_rate']:.3f} to "
          f"{worst_case['custom_rate']:.3f} ({100 * drop:.1f} percentage points).")
    matching_cases.append(dict(worst_case, clip=clip_id, factor=capture_factor,
                               gt_corners=gt_stress[worst_case["frame"]],
                               best_frame=best_case["frame"], best_rate=best_case["custom_rate"]))

print("\nInterpretation: angle is the GT top-edge angle relative to the reference; "
      "scale is sqrt(projected board area / reference area).")
print("Keystone is the largest opposite-edge length ratio (>= 1); "
      "1 means equal opposite lengths. It is a perspective-asymmetry proxy.")
print("Dominant-orientation normalisation helps with in-plane rotation, but the fixed-scale "
      "descriptor does not resize its sampling window or undo perspective deformation.")
print("Use the scale and viewpoint examples above to relate the matching drop to those changes. "
      "Apparent scale also includes foreshortening, so these measurements do not isolate camera distance.")
print("The selected reference/frame pairs remain in matching_cases for the labelled SIFT comparison. "
      "No settings were changed; selected matching figures were saved for the report.")

# Stress examples explain the frozen result; they must not alter Stage 1 settings.
assert config.thresholds()["localisation"] == frozen_localisation


### ABLATION CELL — `cv2.SIFT` baseline (the only cell where SIFT is permitted)
Compare your descriptor's matching rate with SIFT on the same frame pair. Explain the difference in the report.

In [ ]:
sift = cv2.SIFT_create()
def sift_rate(a, b):
    ka, da = sift.detectAndCompute(sp.to_uint8(sp.to_gray(a)), None); kb, db = sift.detectAndCompute(sp.to_uint8(sp.to_gray(b)), None)
    if da is None or db is None or len(db) < 2: return 0.0
    good = [pair[0] for pair in cv2.BFMatcher().knnMatch(da, db, k=2) if len(pair) == 2 and pair[0].distance < 0.8 * pair[1].distance]
    return len(good) / max(len(ka), 1)
worst = rate_indices[int(np.argmin(rates))]
print(f"worst subset frame for your descriptor: {worst} (rate {min(rates):.3f});  SIFT rate on the same pair: {sift_rate(reference, frames[indices.index(worst)]):.3f}")

#### (Our Code) SIFT matching policy and stress-frame comparison

Complete modified copy of the starter block above, followed by our stress-frame comparison. The helper uses the configured ratio threshold and mutual-match policy; subsequent comparisons use this definition.


In [ ]:
sift = cv2.SIFT_create()
def sift_rate(a, b):
    ka, da = sift.detectAndCompute(sp.to_uint8(sp.to_gray(a)), None); kb, db = sift.detectAndCompute(sp.to_uint8(sp.to_gray(b)), None)
    if da is None or db is None or len(db) < 2: return 0.0
    good = [pair[0] for pair in cv2.BFMatcher().knnMatch(da, db, k=2) if len(pair) == 2 and pair[0].distance < config.localisation.matcher.ratio * pair[1].distance]
    # Match frame descriptors back to reference descriptors. Keep a forward
    # match only when the reverse nearest neighbour returns to its original reference point.
    if config.localisation.matcher.mutual:
        reverse = {match.queryIdx: match.trainIdx for match in cv2.BFMatcher().match(db, da)}
        good = [match for match in good if reverse[match.trainIdx] == match.queryIdx]
    return len(good) / max(len(ka), 1)
worst = rate_indices[int(np.argmin(rates))]
print(f"worst subset frame for your descriptor: {worst} (rate {min(rates):.3f});  SIFT rate on the same pair: {sift_rate(reference, frames[indices.index(worst)]):.3f}")
# Measure full SIFT with the same ratio threshold and mutual policy as our matcher.
# Return descriptor counts as well as rates because the two detectors use different denominators.
def sift_measurement(a, b):
    ka, da = sift.detectAndCompute(sp.to_uint8(sp.to_gray(a)), None)
    kb, db = sift.detectAndCompute(sp.to_uint8(sp.to_gray(b)), None)
    if da is None or db is None or len(db) < 2:
        return dict(reference_descriptors=len(ka), frame_descriptors=len(kb), matches=0, rate=0.0)
    nearest = cv2.BFMatcher().knnMatch(da, db, k=2)
    good = [pair[0] for pair in nearest if len(pair) == 2 and
            pair[0].distance < config.localisation.matcher.ratio * pair[1].distance]
    # Match frame descriptors back to reference descriptors. Keep a forward
    # match only when the reverse nearest neighbour returns to its original reference point.
    if config.localisation.matcher.mutual:
        reverse = {match.queryIdx: match.trainIdx for match in cv2.BFMatcher().match(db, da)}
        good = [match for match in good if reverse[match.trainIdx] == match.queryIdx]
    return dict(reference_descriptors=len(ka), frame_descriptors=len(kb),
                matches=len(good), rate=len(good) / max(len(ka), 1))

# Reuse the exact best/worst pairs already selected with frozen Harris settings.
# Do not select different, more favourable frames for the SIFT comparison.
sift_comparisons = []
for case in matching_cases:
    video, _ = data.clip_paths(case["clip"])
    best_image = next(image for index, image in sp.iter_video(video, width=comparison_width,
                       start=case["best_frame"], stop=case["best_frame"] + 1))
    for label, image, index, custom_rate in [("best", best_image, case["best_frame"], case["best_rate"]),
                                            ("worst", case["image"], case["frame"], case["custom_rate"])]:
        custom_gray = sp.to_gray(image)
        points, _ = detect_corners(custom_gray, config.localisation.detector)
        desc, _ = describe_keypoints(custom_gray, points, config.localisation.descriptor)
        # GT geometry describes the image change only; it does not guide either matcher.
        angle, scale, tb, lr, keystone = gt_quad_geometry(data.load_diagnostic_corners(case["clip"], scale=factor)[index])
        row = dict(clip=case["clip"], split="test", width=comparison_width, case=label, frame=index,
                   custom_reference_descriptors=len(d_r), custom_frame_descriptors=len(desc),
                   custom_matches=int(round(custom_rate * len(d_r))), custom_rate=custom_rate,
                   angle_deg=angle, apparent_scale=scale, keystone=keystone, sift=sift_measurement(reference, image))
        sift_comparisons.append(row)
        print(json.dumps(row))
# Rates divide by each method's own reference-descriptor count. This compares
# complete feature pipelines, since SIFT changes both detection and description.
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(sift_comparisons))
ax.bar(x - .18, [row["custom_rate"] for row in sift_comparisons], .36, label="Harris + fixed-scale descriptor")
ax.bar(x + .18, [row["sift"]["rate"] for row in sift_comparisons], .36, label="Full SIFT detector + descriptor")
ax.set_xticks(x, [f"{row['clip']}\n{row['case']} frame {row['frame']}" for row in sift_comparisons], fontsize=8)
ax.set_ylabel("Ratio-test matches / reference descriptors")
ax.set_title(f"Viewpoint and scale test, 640 px; same pairs, ratio {config.localisation.matcher.ratio}, mutual={config.localisation.matcher.mutual}")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES / "sift_comparison.png", bbox_inches="tight")
plt.show()
print("REPORT_SIFT", json.dumps(sift_comparisons))
print("This compares feature pipelines: SIFT changes detector, scale selection and descriptor; it does not isolate scale normalisation alone.")
# Confirm that running the comparison left the frozen localisation unchanged.
assert config.thresholds()["localisation"] == frozen_localisation


## 6. Normalised DLT and conditioning (EXPLORATION)
Build the same system with and without normalisation and compare the singular values of $A$ and the corner-transfer
error. Then construct a degenerate configuration. This is an exploration, not a report question.

In [ ]:
H_true, dst_corners = syn.random_homography(seed=1, frame_size=(640, 360), board_size=(reference.shape[1], reference.shape[0]))
src, dst, _ = syn.synthetic_correspondences(H_true, n_inliers=40, noise=0.3, seed=2,
                                            src_range=(0, reference.shape[1], 0, reference.shape[0]), dst_shape=(360, 640))
H_norm, s_norm, ok = dlt_homography(src, dst)
# the un-normalised system, built here only for the comparison
A = np.zeros((2 * len(src), 9)); x, y = src[:, 0], src[:, 1]; xp, yp = dst[:, 0], dst[:, 1]
A[0::2, 0] = -x; A[0::2, 1] = -y; A[0::2, 2] = -1; A[0::2, 6] = xp * x; A[0::2, 7] = xp * y; A[0::2, 8] = xp
A[1::2, 3] = -x; A[1::2, 4] = -y; A[1::2, 5] = -1; A[1::2, 6] = yp * x; A[1::2, 7] = yp * y; A[1::2, 8] = yp
_, s_raw, vt = np.linalg.svd(A); H_raw = vt[-1].reshape(3, 3); H_raw /= H_raw[2, 2]
ref_c = sp.rect_corners(reference.shape[1], reference.shape[0])
for name, H, sv in [("normalised", H_norm, s_norm), ("un-normalised", H_raw, s_raw)]:
    print(f"{name:14s} corner error {metrics.corner_error(transfer_points(H, ref_c), dst_corners) * 100:.4f}%   "
          f"condition number sigma_1/sigma_8 = {sv[0] / sv[7]:.3g}")
print("With float64 SVD both solve this small, noise-free-ish system; the conditioning gap is what grows with coordinate magnitude, noise and outliers.")
viz.plot_singular_values([s_norm, s_raw], ["normalised", "un-normalised"], "singular values of A"); plt.show()
# degenerate: all correspondences inside a small patch of the board
patch_src, patch_dst, _ = syn.synthetic_correspondences(H_true, n_inliers=30, noise=0.3, seed=3, src_range=(600, 680, 150, 180))
H_patch, s_patch, ok_patch = dlt_homography(patch_src, patch_dst)
print("small-patch configuration: status", ok_patch, "| corner error %.1f%%" % (metrics.corner_error(transfer_points(H_patch, ref_c), dst_corners) * 100 if H_patch is not None else float("nan")))
coll = np.stack([np.linspace(0, 400, 6), np.linspace(0, 100, 6)], 1); print("collinear configuration: status", dlt_homography(coll, coll * 2)[2])

### EXPLORATION — when does normalisation matter, and what does a degenerate configuration look like?
Scaffold: shift the correspondences to large coordinates (e.g. add 4000 px, as in a 4K frame) and add noise, and watch the
condition number and the corner error of the un-normalised system diverge from the normalised one. Then make three of the
four points collinear plus one more and inspect the singular values: the second-smallest one tells you that the solution
is not unique.

In [ ]:
def raw_dlt(src, dst):
    A = np.zeros((2 * len(src), 9)); x, y = src[:, 0], src[:, 1]; xp, yp = dst[:, 0], dst[:, 1]
    A[0::2, 0] = -x; A[0::2, 1] = -y; A[0::2, 2] = -1; A[0::2, 6] = xp * x; A[0::2, 7] = xp * y; A[0::2, 8] = xp
    A[1::2, 3] = -x; A[1::2, 4] = -y; A[1::2, 5] = -1; A[1::2, 6] = yp * x; A[1::2, 7] = yp * y; A[1::2, 8] = yp
    _, sv, vt = np.linalg.svd(A); H = vt[-1].reshape(3, 3); return H / H[2, 2], sv

print(f"{'offset':>7s} {'noise':>6s} {'cond (norm)':>12s} {'cond (raw)':>11s} {'err norm':>9s} {'err raw':>8s}")
for offset in (0.0, 2000.0, 8000.0):
    for noise in (0.3, 2.0):
        s2, d2, _ = syn.synthetic_correspondences(H_true, n_inliers=12, noise=noise, seed=5, src_range=(0, reference.shape[1], 0, reference.shape[0]), dst_shape=(360, 640))
        Hn, svn, _ = dlt_homography(s2 + offset, d2 + offset); Hr, svr = raw_dlt(s2 + offset, d2 + offset)
        en = metrics.corner_error(transfer_points(Hn, ref_c + offset) - offset, dst_corners) * 100
        er = metrics.corner_error(transfer_points(Hr, ref_c + offset) - offset, dst_corners) * 100
        print(f"{offset:7.0f} {noise:6.1f} {svn[0]/svn[7]:12.3g} {svr[0]/svr[7]:11.3g} {en:8.3f}% {er:7.3f}%")
# degenerate: three collinear points + one more (a 4-point minimal set with no unique solution)
deg_src = np.array([[0, 0], [100, 0], [200, 0], [50, 80]], float); deg_dst = transfer_points(H_true, deg_src)
Hd, sd, okd = dlt_homography(deg_src, deg_dst)
print("3 collinear + 1: status", okd, "| two smallest singular values", sd[-2:], "(both ~0 => the null space is 2-dimensional)")
# YOUR CODE HERE (optional) — plot the full singular-value spectra of the degenerate and the generic 4-point systems

#### (Our Code) DLT conditioning and singular-value exploration

Complete modified copy of the starter cell above, with the corrected degeneracy diagnostic and our singular-value exploration. Running both cells repeats the conditioning table and shows both diagnostic messages; the fixed seeds keep the final calculations unchanged.


In [ ]:
def raw_dlt(src, dst):
    A = np.zeros((2 * len(src), 9)); x, y = src[:, 0], src[:, 1]; xp, yp = dst[:, 0], dst[:, 1]
    A[0::2, 0] = -x; A[0::2, 1] = -y; A[0::2, 2] = -1; A[0::2, 6] = xp * x; A[0::2, 7] = xp * y; A[0::2, 8] = xp
    A[1::2, 3] = -x; A[1::2, 4] = -y; A[1::2, 5] = -1; A[1::2, 6] = yp * x; A[1::2, 7] = yp * y; A[1::2, 8] = yp
    _, sv, vt = np.linalg.svd(A); H = vt[-1].reshape(3, 3); return H / H[2, 2], sv

print(f"{'offset':>7s} {'noise':>6s} {'cond (norm)':>12s} {'cond (raw)':>11s} {'err norm':>9s} {'err raw':>8s}")
for offset in (0.0, 2000.0, 8000.0):
    for noise in (0.3, 2.0):
        s2, d2, _ = syn.synthetic_correspondences(H_true, n_inliers=12, noise=noise, seed=5, src_range=(0, reference.shape[1], 0, reference.shape[0]), dst_shape=(360, 640))
        Hn, svn, _ = dlt_homography(s2 + offset, d2 + offset); Hr, svr = raw_dlt(s2 + offset, d2 + offset)
        en = metrics.corner_error(transfer_points(Hn, ref_c + offset) - offset, dst_corners) * 100
        er = metrics.corner_error(transfer_points(Hr, ref_c + offset) - offset, dst_corners) * 100
        print(f"{offset:7.0f} {noise:6.1f} {svn[0]/svn[7]:12.3g} {svr[0]/svr[7]:11.3g} {en:8.3f}% {er:7.3f}%")
# degenerate: three collinear points + one more (a 4-point minimal set with no unique solution)
deg_src = np.array([[0, 0], [100, 0], [200, 0], [50, 80]], float); deg_dst = transfer_points(H_true, deg_src)
Hd, sd, okd = dlt_homography(deg_src, deg_dst)
print("3 collinear + 1: status", okd, "| smallest returned singular value", sd[-1], "(plus the implicit zero => nullity 2)")
# YOUR CODE HERE (optional) — plot the full singular-value spectra of the degenerate and the generic 4-point systems

# Compare the eight singular values returned for each normalised 8 x 9 system.
# SVD omits the ninth (implicit zero) value: every such system has a null direction.
generic_src = ref_c.copy()
generic_dst = transfer_points(H_true, generic_src)
H_generic, s_generic, ok_generic = dlt_homography(generic_src, generic_dst)
spectrum_cases = [("Generic four corners", s_generic, ok_generic),
                  ("Three collinear + one", sd, okd)]

fig, ax = plt.subplots(figsize=(9, 4))
print(f"{'configuration':25s} {'rank(A)':>7s} {'nullity':>7s} {'DLT status':>10s}")
for label, singular_values, status in spectrum_cases:
    relative = singular_values / singular_values[0]
    # Use the same numerical rank tolerance as dlt_homography.
    relative_tol = 9 * np.finfo(float).eps
    rank = int(np.count_nonzero(relative > relative_tol))
    nullity = 9 - rank
    # Clip only for the log plot; rank and the printed spectrum use actual values.
    ax.semilogy(np.arange(1, len(relative) + 1),
                np.maximum(relative, np.finfo(float).eps), "o-",
                label=f"{label}: rank {rank}, nullity {nullity}")
    print(f"{label:25s} {rank:7d} {nullity:7d} {str(status):>10s}")
    print("  singular values:", np.array2string(singular_values, precision=4))
ax.axhline(relative_tol, color="black", ls="--", lw=1, label="Numerical rank tolerance")
ax.set_xticks(np.arange(1, 9))
ax.set_xlabel("Singular-value index (descending order)")
ax.set_ylabel("Singular value / largest singular value")
ax.set_title("Normalised DLT: generic versus degenerate four-point systems")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
plt.show()
print("A generic 8 x 9 system has rank 8 and one null direction: H is unique up to scale.")
print("The collinear case has an additional near-zero returned singular value, "
      "so its null space has dimension 2 and H is not uniquely determined.")
print("Only eight singular values are returned for an 8 x 9 matrix; "
      "the implicit ninth zero is not plotted. Values are normalised for comparison.")


### SANITY-CHECK CELL — `cv2.findHomography` (the only cell where it is permitted)

In [ ]:
H_cv, _ = cv2.findHomography(src.astype(np.float32), dst.astype(np.float32), 0)
print("max |H_ours - H_cv| after normalising H[2,2]:", np.abs(H_norm / H_norm[2, 2] - H_cv / H_cv[2, 2]).max())

## 7. RANSAC: inlier threshold, iterations, and $N(p,\varepsilon)$ (EXPLORATION)
Vary the threshold and the iteration cap on correspondences with injected outliers, and relate the inlier count and the
corner error to the required iteration count. The residual histogram is the evidence you should show for any failure
case in the report; the sweep itself is an exploration, not a report question.

In [ ]:
src_o, dst_o, is_in = syn.synthetic_correspondences(H_true, n_inliers=60, n_outliers=60, noise=0.5, seed=4,
                                                   src_range=(0, reference.shape[1], 0, reference.shape[0]), dst_shape=(360, 640))
print(f"{'thr':>5s} {'max_it':>6s} {'used':>5s} {'inliers':>7s} {'corner err':>10s}")
for thr in (2.0, 4.0, 9.0, 25.0):
    for max_it in (20, 200, 2000):
        H, inl, resid, used, ok = ransac_homography(src_o, dst_o, replace(config.localisation.ransac, threshold=thr, max_iterations=max_it, seed=0))
        e = metrics.corner_error(transfer_points(H, ref_c), dst_corners) * 100 if H is not None else float("nan")
        print(f"{thr:5.1f} {max_it:6d} {used:5d} {int(inl.sum()):7d} {e:9.3f}%")
H, inl, resid, used, ok = ransac_homography(src_o, dst_o, config.localisation.ransac)
viz.plot_residual_histogram(resid, inl, config.localisation.ransac.threshold, "symmetric transfer error per correspondence"); plt.show()
eps = np.linspace(0.05, 0.8, 16); N = np.log(1 - 0.995) / np.log(1 - (1 - eps) ** 4)
plt.figure(figsize=(5, 3)); plt.semilogy(eps, N); plt.xlabel("outlier fraction eps"); plt.ylabel("N(p=0.995, eps)"); plt.grid(alpha=.3); plt.show()

### EXPLORATION — does the adaptive iteration count match $N(p,\varepsilon)$?
Scaffold: for several true outlier fractions, run RANSAC with a large iteration cap and compare the number of iterations it
*used* with $N(p,\varepsilon)$ computed from the inlier fraction it *found*; then check how often the result is a success
over repeated seeds when the cap is below $N$.

In [ ]:
def N_required(p, eps, s=4):
    return int(np.ceil(np.log(1 - p) / np.log(1 - (1 - eps) ** s)))

print(f"{'true eps':>8s} {'found eps':>9s} {'N(0.995)':>8s} {'used':>5s} {'success@cap=N/4':>15s}")
for n_out in (20, 60, 120, 180):
    s3, d3, in3 = syn.synthetic_correspondences(H_true, n_inliers=60, n_outliers=n_out, noise=0.5, seed=7, src_range=(0, reference.shape[1], 0, reference.shape[0]), dst_shape=(360, 640))
    H, inl, resid, used, ok = ransac_homography(s3, d3, replace(config.localisation.ransac, max_iterations=5000, seed=0))
    eps_found = 1 - inl.mean(); N = N_required(0.995, eps_found)
    succ = []
    for seed in range(10):
        Hs, *_ = ransac_homography(s3, d3, replace(config.localisation.ransac, max_iterations=max(1, N // 4), seed=seed))
        succ.append(Hs is not None and metrics.corner_error(transfer_points(Hs, ref_c), dst_corners) < 0.01)
    print(f"{n_out/(60+n_out):8.2f} {eps_found:9.2f} {N:8d} {used:5d} {np.mean(succ):15.0%}")
# YOUR CODE HERE (optional) — repeat with threshold 2 / 9 / 25 px and explain why the found inlier fraction moves with the threshold


#### (Our Code) Exploration: RANSAC thresholds and reduced iteration budgets


In [ ]:

# Thresholds are in px^2: symmetric_transfer_error sums squared errors in both directions.
threshold_rows = []
threshold_values = (2.0, 9.0, 25.0)
confidence = config.localisation.ransac.confidence
short_run_seeds = range(10)
print("\nThreshold exploration on identical synthetic matches (data seed 7, noise 0.5 px).")
print("Only the per-run RANSAC threshold changes; the pipeline configuration stays fixed.")
print(f"{'thr px^2':>8s} {'true eps':>8s} {'found eps':>9s} {'N(p)':>7s} "
      f"{'used':>5s} {'precision':>9s} {'recall':>7s} {'err %':>8s} "
      f"{'status':>6s} {'cap':>5s} {'success@cap':>11s}")
for n_out in (20, 60, 120, 180):
    # Reuse each correspondence set at every threshold for a controlled comparison.
    threshold_src, threshold_dst, true_inliers = syn.synthetic_correspondences(
        H_true, n_inliers=60, n_outliers=n_out, noise=0.5, seed=7,
        src_range=(0, reference.shape[1], 0, reference.shape[0]), dst_shape=(360, 640),
    )
    for threshold in threshold_values:
        run_config = replace(config.localisation.ransac, threshold=threshold,
                             max_iterations=5000, seed=0)
        H_trial, trial_inliers, _, trial_used, trial_ok = ransac_homography(
            threshold_src, threshold_dst, run_config)
        found_fraction = float(trial_inliers.mean())
        found_eps = 1 - found_fraction
        # Handle all-inlier / no-inlier limits without taking log(0) or dividing by zero.
        if found_fraction == 0:
            required = float("inf")
        elif found_fraction == 1:
            required = 1
        else:
            required = N_required(confidence, found_eps)
        short_cap = (max(1, required // 4) if np.isfinite(required)
                     else run_config.max_iterations)
        short_cap = min(short_cap, run_config.max_iterations)
        successes = []
        for trial_seed in short_run_seeds:
            H_short, _, _, _, short_ok = ransac_homography(
                threshold_src, threshold_dst,
                replace(run_config, max_iterations=short_cap, seed=trial_seed),
            )
            successes.append(bool(short_ok and H_short is not None and
                metrics.corner_error(transfer_points(H_short, ref_c), dst_corners) < 0.01))
        true_positives = int(np.count_nonzero(trial_inliers & true_inliers))
        precision = true_positives / max(int(trial_inliers.sum()), 1)
        recall = true_positives / int(true_inliers.sum())
        corner_err = (metrics.corner_error(transfer_points(H_trial, ref_c), dst_corners)
                      if trial_ok and H_trial is not None else float("inf"))
        row = dict(threshold=threshold, true_eps=n_out / len(true_inliers),
                   found_eps=found_eps, required=required, used=trial_used,
                   precision=precision, recall=recall, corner_error=corner_err,
                   short_cap=short_cap, short_success=float(np.mean(successes)), status=trial_ok)
        threshold_rows.append(row)
        print(f"{threshold:8.1f} {row['true_eps']:8.2f} {found_eps:9.2f} {required:7.0f} "
              f"{trial_used:5d} {precision:9.2f} {recall:7.2f} {100 * corner_err:8.3f} "
              f"{str(trial_ok):>6s} {short_cap:5d} {row['short_success']:11.0%}")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for threshold in threshold_values:
    rows = [row for row in threshold_rows if row['threshold'] == threshold]
    true_eps = [row['true_eps'] for row in rows]
    label = f"threshold = {threshold:g} px^2"
    axes[0].plot(true_eps, [1 - row['found_eps'] for row in rows], "o-", label=label)
    axes[1].plot(true_eps, [row['used'] for row in rows], "o-", label=label)
    axes[2].plot(true_eps, [row['short_success'] for row in rows], "o-", label=label)
true_eps = sorted({row['true_eps'] for row in threshold_rows})
axes[0].plot(true_eps, 1 - np.array(true_eps), "k--", label="True inlier fraction")
axes[0].set_ylabel("Found inlier fraction")
axes[0].set_ylim(0, 1.05)
axes[1].set_ylabel("Iterations used (cap 5000)")
axes[1].set_yscale("log")
axes[2].set_ylabel("Success fraction at reduced cap")
axes[2].set_ylim(-0.05, 1.05)
for ax in axes:
    ax.set_xlabel("True outlier fraction")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("RANSAC threshold study: fixed matches, ten seeds per reduced-cap run")
fig.tight_layout()
plt.show()
print("A tighter threshold can reject noisy true matches, lowering recall and the found "
      "inlier fraction; a looser threshold can retain them but may admit incorrect matches.")
print("For a fixed candidate H, raising the threshold can only add inliers. Across RANSAC "
      "runs, the selected model and adaptive stopping also change, so results need not be monotonic.")
print("A larger found inlier fraction reduces the predicted iteration budget. "
      "N(p) is a sampling-confidence calculation, not a guarantee of accurate corners.")
print("Reduced caps are computed separately from each threshold's fitted inlier fraction; "
      "success here requires a valid estimate with normalised corner error below 0.01.")


## 8. Warping convention check
A synthetic checkerboard warped with a known homography must land on the expected corners. State in your report whether
`x` indexes columns and where your pixel centres sit.

In [ ]:
cb = np.repeat(syn.checkerboard(4, 16, 8)[..., None], 3, -1)   # 32 x 128 artwork
target = np.array([[60, 40], [300, 70], [290, 150], [70, 140]], float)
H_cb = syn.homography_from_corners(sp.rect_corners(128, 32), target)
warped, cov = warp_artwork(cb, H_cb, (200, 360, 3))
ys, xs = np.nonzero(cov)
print("coverage bbox x:", xs.min(), xs.max(), " y:", ys.min(), ys.max(), "| expected approx", target[:, 0].min(), target[:, 0].max(), target[:, 1].min(), target[:, 1].max())


#### (Our Code) Warp-convention figure and numerical evidence

The plotting statement retains the starter figure setup and display, with a replacement title and an added figure export. The report measurements and their comment below are additions.


In [ ]:
plt.figure(figsize=(6, 3.5)); plt.imshow(viz.draw_quad(warped, target)); plt.axis("off"); plt.title("Synthetic checkerboard, known H; integer pixel centres, x = columns"); plt.savefig(FIGURES / "warp_convention.png", bbox_inches="tight"); plt.show()
# Record pixel-centre bounds and the supplied corner metric for the convention check.
print("REPORT_WARP", json.dumps(dict(coverage_bbox=[int(xs.min()), int(xs.max()), int(ys.min()), int(ys.max())],
                                     corner_error=metrics.corner_error(transfer_points(H_cb, sp.rect_corners(128, 32)), target))))


### COMPARISON CELL — `cv2.warpPerspective` (the only cell where it is permitted)

In [ ]:
ref_cv = cv2.warpPerspective(cb, H_cb.astype(np.float32), (360, 200))
print("mean |ours - cv2| inside coverage:", float(np.abs(warped - ref_cv)[cov].mean()))

## Reflection prompts
1. Which NMS mode gives better spatial coverage of the board, and at what cost in repeatability?
2. On which frames does the ratio test discard most of the correct matches, and why?
3. What does the second-smallest singular value of $A$ tell you that the smallest does not?
4. On the viewpoint and scale clips, which part of the change does your descriptor survive, and which part does SIFT survive that yours does not (**Question 1**)?